# Preparación de ERA5 diario para todas las provincias (uso del docente)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/crispaliz/cambio-climatico-espam/blob/main/herramientas/reanalisis/preparar_era5.ipynb)

1. Descarga del Climate Data Store los estadísticos diarios de ERA5 para el recuadro de Ecuador (todas las provincias, incluida Galápagos), con el día en hora local (UTC-5).
2. Reparte el resultado en **un archivo por provincia** con todo el periodo, que se publica en el release `era5-v1`. Cada estudiante descarga solo el de su provincia.

**Solicitudes al servidor:** 4 por año (las variables con el mismo estadístico van juntas), en serie, con reintentos y sin repetir lo ya descargado. Para 1981–2024 son 176 solicitudes; puede tomar uno o varios días según la cola del CDS. Es reanudable.

Requiere el API Token en Colab › Secrets como `CDSAPI_KEY` y la licencia aceptada en *ERA5 post-processed daily statistics on single levels from 1940 to present*.


| Variable | Columna | Unidad | ERA5 | CORe |
|---|---|---|---|---|
| Temperatura media, máxima, mínima a 2 m | `tmed`, `tmax`, `tmin` | °C | Estadísticos diarios de `2m_temperature` | Media de TMP 2 m; máx. de TMAX; mín. de TMIN |
| Precipitación | `pr` | mm/día | Suma diaria de `total_precipitation` | Suma de PRATE × 21 600 s |
| Humedad relativa | `hr` | % | De T y punto de rocío medios (Magnus) | De SPFH, T y presión de cada 6 h, promediada |
| Humedad específica | `q` | g/kg | De punto de rocío y presión medios | SPFH a 2 m, promediada |
| Presión en superficie | `ps` | hPa | Media de `surface_pressure` | Media de PRES sfc |
| Viento a 10 m | `u10`, `v10` | m/s | Medias de las componentes | Medias de UGRD y VGRD |


© 2026 Cristian Omar Paliz Acosta · ESPAM MFL · Todos los derechos reservados

In [ ]:
!pip -q install cdsapi regionmask

import os, sys, pathlib, subprocess
REPO_DIR = pathlib.Path(os.environ.get("CC_REPO", "/content/cambio-climatico-espam"))
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/crispaliz/cambio-climatico-espam", str(REPO_DIR)], check=True)
sys.path.insert(0, str(REPO_DIR))
from herramientas.reanalisis import era5, core, release
import xarray as xr
from google.colab import drive
drive.mount("/content/drive")

TRABAJO = pathlib.Path("/content/drive/MyDrive/ERA5_Ecuador/descargas")   # descargas del CDS por año y estadístico
ANUAL = pathlib.Path("/content/drive/MyDrive/ERA5_Ecuador/anual")         # un archivo por año, todo Ecuador
RELEASE = pathlib.Path("/content/drive/MyDrive/ERA5_Ecuador/release")     # un archivo por provincia
CDS_TOKEN = ""   # su API Token personal; vacío si lo guardó en Secrets como CDSAPI_KEY
print("Credencial activa:", era5.credenciales(CDS_TOKEN))
print("Recuadro de Ecuador [N, O, S, E]:", era5.RECUADRO_ECUADOR)

## 1. Prueba con un año

In [ ]:
f = era5.preparar_diario([2000], TRABAJO, ANUAL)[0]
ds = xr.open_dataset(f); print(ds)
print({v: round(float(ds[v].mean()), 2) for v in ds.data_vars})
ds["pr"].sel(lat=slice(-2.5, 1), lon=slice(-81, -79)).mean(["lat", "lon"]).plot(figsize=(10, 3))

Revise antes de seguir: 366 días; Tmax > Tmed > Tmin; precipitación en mm/día; HR entre 0 y 100 %; q de unos 15–20 g/kg en la Costa y menos en la Sierra; q entre ~5 y 20 g/kg en la Costa y menor en la Sierra; presión en hPa (≈1010 en la costa, ≈700 en la Sierra alta); viento de pocos m/s.

Si una solicitud falla por el nombre de un campo, compárelo con la pestaña **Download** del conjunto (opción "Show API request") y ajuste `era5.solicitud_diaria` o `era5.GRUPOS_DIARIOS`. Si el CDS rechaza varias variables en una misma solicitud, deje una variable por grupo.

## 2. Periodo completo

In [ ]:
ANIOS = range(1981, 2025)
anuales = era5.preparar_diario(ANIOS, TRABAJO, ANUAL)
print(len(anuales), "años listos")

## 3. Un archivo por provincia

In [ ]:
archivos = release.por_provincia("era5", anuales, RELEASE)
for f in archivos: print(f"{f.name:45s} {f.stat().st_size / 1e6:6.1f} MB")

## 4. Publicar

En GitHub: **Releases › Draft a new release**, etiqueta `era5-v1`, y adjunte los `era5_diario_<provincia>.nc` de `Mi unidad/ERA5_Ecuador/release`. En la descripción incluya la cita de ERA5 y el aviso de Copernicus que exige su licencia ("Contains modified Copernicus Climate Change Service information [año]").